In [ ]:
import anndata as ad
import scipy.sparse as sp
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [ ]:
DATASET = "retinal"   # swap to "pbmc", "seaad_paired", "seaad_unpaired"
BASE    = f"../../../data/processed/{DATASET}"

# 1. Load files
rna      = ad.read_h5ad(f"{BASE}_rna_sub.h5ad")       # (400 cells, 500 genes)
knn      = np.load(f"{BASE}_knn_indices.npy")          # (400, 20)  0-based
triplets = pd.read_csv(f"{BASE}_triplets.csv")         # target_gene | peak | spearman_r | TF
labels   = pd.read_csv(f"{BASE}_gene_labels.csv")      # gene | type | associated_peaks | associated_TFs


In [ ]:
peak_matrix   = np.load(f"../../../data/processed/peak_matrix.npy")

In [ ]:
X = rna.X.toarray() if sp.issparse(rna.X) else rna.X  # (400, 500)
expr_df = pd.DataFrame(
    X.T,                        # → (500 genes, 400 cells)
    index   = rna.var_names,    # 500 gene names
    columns = rna.obs['_original_rna_cell'],    # 400 cell barcodes
)

In [ ]:
# 3. peak_df: peaks × cells
# peak_matrix is (400 cells, 217 peaks) — transpose to (217 peaks, 400 cells)
# Peak names come from unique peaks in triplets (217 unique values)
peak_names = triplets["peak"].unique()                 # 217 peak names
peak_df = pd.DataFrame(
    peak_matrix.T,              # → (217 peaks, 400 cells)
    index   = peak_names,
    columns = rna.obs_names,
)

In [ ]:
rows = []
for _, row in labels.iterrows():
    gene  = row["gene"]
    gtype = row["type"]          # "TF" or "target"

    if pd.isna(row["associated_peaks"]):
        # Remove empt peak rows
        continue
    else:
        peaks_for_gene = str(row["associated_peaks"]).split(",")
        TF_str = (
            ";".join(str(row["associated_TFs"]).split(","))
            if pd.notna(row["associated_TFs"]) else ""
        )
        for peak in peaks_for_gene:
            rows.append({"gene": gene, "peak": peak.strip(), "TF": TF_str, "label": gtype})

labs_df = pd.DataFrame(rows)

In [ ]:
labs = GenePeakOverlapLabs(
    genes  = labs_df["gene"].tolist(),
    peaks  = labs_df["peak"].tolist(),
    TFs    = labs_df["TF"].tolist(),      # semicolon-separated TF names
    labels = labs_df["label"].tolist(),   # "TF" or "target"
)

# # Sanity checks
# assert expr_df.shape == (500, 400), f"Unexpected expr shape: {expr_df.shape}"
# assert peak_df.shape == (217, 400), f"Unexpected peak shape: {peak_df.shape}"
# assert knn.shape     == (400, 20),  f"Unexpected KNN shape: {knn.shape}"

print(f"expr_matrix : {expr_df.shape}  (genes × cells)")
print(f"peak_df     : {peak_df.shape}  (peaks × cells)")
print(f"knn         : {knn.shape}  (cells × neighbors)")
print(f"labs entries: {len(labs.genes)} gene-peak rows")
print(f"TF genes    : {(labels['type'] == 'TF').sum()}, "
        f"target genes: {(labels['type'] == 'target').sum()}")

In [ ]:
# Run wScReNI
networks = infer_wScReNI_c_networks(
    expr_matrix              = expr_df,
    gene_peak_overlap_matrix = peak_df,
    gene_peak_overlap_labs   = labs,
    nearest_neighbors_idx    = knn,       # (400, 20), 0-based
    network_path             = "output/networks",
    data_name                = DATASET,
    cell_index               = None,      # None = all 400 cells
    nthread                  = 8,
    max_cell_per_batch       = 10,
)


In [ ]:
# 7. Output
# networks is a plain list of length 400
# networks[i] is a np.ndarray of shape (500, 500) for cell i
# networks[i][row, col] = regulatory weight of gene col → gene row in cell i
#
# Files are also saved to: output/networks/wScReNI/{i}.{cell_barcode}.network.txt

# Optional: convert to dict keyed by cell barcode for easier lookup
cell_barcodes = rna.obs_names.tolist()
networks_dict = {cell_barcodes[i]: networks[i] for i in range(len(networks))}

# Optional: save results back into AnnData
rna.uns["wScReNI_networks"] = networks
rna.write_h5ad(f"output/{DATASET}_with_networks.h5ad")

print(f"\nDone. {len(networks)} networks inferred.")
print(f"Network shape per cell : {networks[0].shape}")   # (500, 500)
print(f"Output dir             : output/networks/wScReNI/")

In [ ]:
from .calculate_scnetwork_precision_recall import calculate_scnetwork_precision_recall
precision_recall_scnetworks=calculate_scnetwork_precision_recall(networks_dict,TF_str,gene)